In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import random
from io import StringIO

# 1. Conecta ao Google Drive e libera a visualização de todas as colunas
drive.mount('/content/drive')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.expand_frame_repr', False)

caminho_arquivo = '/content/drive/MyDrive/ENEM_2019/MICRODADOS_ENEM_2019.csv'

# 2. Função com sorteio aleatório fixo (seed=42) e todas as suas regras de filtro
def pegar_linhas_aleatorias(caminho, n=10000, seed=42):
    random.seed(seed)  # Garante que os 1.000 sorteados sejam sempre os mesmos
    notas = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']

    mapa_idade = {
        1: 15, 2: 17, 3: 18, 4: 19, 5: 20, 6: 21, 7: 22, 8: 23, 9: 24, 10: 25,
        11: 28, 12: 33, 13: 38, 14: 43, 15: 48, 16: 53, 17: 58, 18: 63, 19: 68, 20: 70
    }

    mapa_ano = {
        0: 0,
        1: 2018, 2: 2017, 3: 2016, 4: 2015, 5: 2014, 6: 2013,
        7: 2012, 8: 2011, 9: 2010, 10: 2009, 11: 2008, 12: 2007,
        13: 0
    }

    ufs = [
        'AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO',
        'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR',
        'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO'
    ]

    status_redacao = [
        (1, 'REDACAO_OK'),
        (2, 'REDACAO_ANULADA'),
        (3, 'REDACAO_COPIA'),
        (4, 'REDACAO_BRANCO'),
        (6, 'REDACAO_FUGA'),
        (7, 'REDACAO_TIPO_TEXTUAL_ERRADO'),
        (8, 'REDACAO_INSUFICIENTE'),
        (9, 'REDACAO_DESCONECTADA')
    ]

    escolaridade_map = [
        ('A', 'NUNCA_ESTUDOU'),
        ('B', 'FUND1_INCOMPLETO'),
        ('C', 'FUND2_INCOMPLETO'),
        ('D', 'MEDIO_INCOMPLETO'),
        ('E', 'SUPERIOR_INCOMPLETO'),
        ('F', 'SUPERIOR_COMPLETO'),
        ('G', 'POS_GRADUACAO')
    ]

    # trabalho_map = [
    #     ('A', 'TRABALHO_GRUPO_1'),
    #     ('B', 'TRABALHO_GRUPO_2'),
    #     ('C', 'TRABALHO_GRUPO_3'),
    #     ('D', 'TRABALHO_GRUPO_4'),
    #     ('E', 'TRABALHO_GRUPO_5')
    # ]



    mapa_trabalho_salario = {
        'A': 1119.0,
        'B': 1350.0,
        'C': 1800.0,
        'D': 3050.0,
        'E': 5450.0,
        'F': 0.0
    }

    mapa_renda = {
        'A': 0.0,
        'B': 499.0,
        'C': 1247.51,
        'D': 1746.51,
        'E': 2245.51,
        'F': 2744.51,
        'G': 3493.01,
        'H': 4491.01,
        'I': 5489.01,
        'J': 6487.01,
        'K': 7485.01,
        'L': 8483.01,
        'M': 9481.01,
        'N': 10978.01,
        'O': 13473.01,
        'P': 17465.0,
        'Q': 21000.0
    }

    mapa_q007 = {
        'A': 0.0,
        'B': 1.5,
        'C': 3.5,
        'D': 6.0
    }

    mapa_qtd_itens = {
        'A': 0,
        'B': 1,
        'C': 2,
        'D': 3,
        'E': 5
    }

    renomear_qtd = {
        'Q008': 'QTD_BANHEIROS',
        'Q009': 'QTD_QUARTOS_DE_DORMIR',
        'Q010': 'QTD_CARRO',
        'Q011': 'QTD_MOTO',
        'Q012': 'QTD_GELADEIRA',
        'Q013': 'QTD_FREEZER',
        'Q014': 'QTD_LAVA_ROUPA',
        'Q015': 'QTD_SECA_ROUPA',
        'Q016': 'QTD_MICROONDAS',
        'Q017': 'QTD_LAVA_LOUCA',
        'Q019': 'QTD_TV',
        'Q022': 'QTD_CELULAR',
        'Q024': 'QTD_COMPUTADOR'
    }

    mapa_binario = {'A': 0, 'B': 1}

    renomear_tem = {
        'Q018': 'TEM_ASPIRADOR_DE_PO',
        'Q020': 'TEM_DVD',
        'Q021': 'TEM_TV_ASSINATURA',
        'Q023': 'TEM_TELEFONE_FIXO',
        'Q025': 'TEM_INTERNET'
    }

    with open(caminho, 'r', encoding='iso-8859-1') as f:
        cabecalho = f.readline()

        # Localiza a posição exata das 5 colunas de notas no cabeçalho
        colunas = cabecalho.strip().split(';')
        indices_notas = [colunas.index(nota) for nota in notas]
        idx_mt = colunas.index('NU_NOTA_MT')

        linhas_sorteadas = []
        validos = 0

        for linha in f:
            campos = linha.split(';')

            # Pula a linha antes do sorteio se qualquer uma das 5 notas estiver vazia
            if any(campos[idx] == '' for idx in indices_notas):
                continue

            # Pula a linha se a nota de Matemática for zero
            if float(campos[idx_mt]) == 0:
                continue

            # Sorteio justo (Reservoir Sampling) apenas entre os presentes
            if len(linhas_sorteadas) < n:
                linhas_sorteadas.append(linha)
            else:
                j = random.randint(0, validos)
                if j < n:
                    linhas_sorteadas[j] = linha

            validos += 1

    conteudo = [cabecalho] + linhas_sorteadas

    # Monta o DataFrame já descartando as colunas indesejadas
    df = pd.read_csv(
        StringIO(''.join(conteudo)),
        sep=';',
        usecols=lambda col: (
            not col.startswith(('TX_', 'NU_NOTA_COMP', 'CO_', 'TP_PRESENCA_'))
            and not col.endswith('_ESC')
        )
    )

    df['TP_FAIXA_ETARIA'] = df['TP_FAIXA_ETARIA'].map(mapa_idade)

    df['TP_SEXO'] = df['TP_SEXO'].map({'F': 1, 'M': 0})
    df = df.rename(columns={'TP_SEXO': 'IS_FEM'})

    df['TP_ANO_CONCLUIU'] = df['TP_ANO_CONCLUIU'].map(mapa_ano)

    # pos = df.columns.get_loc('TP_ESTADO_CIVIL') + 1
    # df.insert(pos,     'IS_SOLTEIRO',   (df['TP_ESTADO_CIVIL'] == 1).astype(int))
    # df.insert(pos + 1, 'IS_CASADO',     (df['TP_ESTADO_CIVIL'] == 2).astype(int))
    # df.insert(pos + 2, 'IS_DIVORCIADO', (df['TP_ESTADO_CIVIL'] == 3).astype(int))
    # df.insert(pos + 3, 'IS_VIUVO',      (df['TP_ESTADO_CIVIL'] == 4).astype(int))
    df = df.drop(columns=['TP_ESTADO_CIVIL'])

    # pos_raca = df.columns.get_loc('TP_COR_RACA') + 1
    # df.insert(pos_raca,     'IS_BRANCA',   (df['TP_COR_RACA'] == 1).astype(int))
    # df.insert(pos_raca + 1, 'IS_PRETA',    (df['TP_COR_RACA'] == 2).astype(int))
    # df.insert(pos_raca + 2, 'IS_PARDA',    (df['TP_COR_RACA'] == 3).astype(int))
    # df.insert(pos_raca + 3, 'IS_AMARELA',  (df['TP_COR_RACA'] == 4).astype(int))
    # df.insert(pos_raca + 4, 'IS_INDIGENA', (df['TP_COR_RACA'] == 5).astype(int))
    df = df.drop(columns=['TP_COR_RACA'])

    # pos_nac = df.columns.get_loc('TP_NACIONALIDADE') + 1
    # df.insert(pos_nac,     'IS_BRASILEIRO',   (df['TP_NACIONALIDADE'] == 1).astype(int))
    # df.insert(pos_nac + 1, 'IS_NATURALIZADO', (df['TP_NACIONALIDADE'] == 2).astype(int))
    # df.insert(pos_nac + 2, 'IS_ESTRANGEIRO',  (df['TP_NACIONALIDADE'] == 3).astype(int))
    # df.insert(pos_nac + 3, 'IS_BR_EXTERIOR',  (df['TP_NACIONALIDADE'] == 4).astype(int))
    df = df.drop(columns=['TP_NACIONALIDADE'])

    # pos_conc = df.columns.get_loc('TP_ST_CONCLUSAO') + 1
    # df.insert(pos_conc,     'IS_EM_COMP',   (df['TP_ST_CONCLUSAO'] == 1).astype(int))
    # df.insert(pos_conc + 1, 'EM_COMP_2019', (df['TP_ST_CONCLUSAO'] == 2).astype(int))
    # df.insert(pos_conc + 2, 'IN_EM',           (df['TP_ST_CONCLUSAO'] == 3).astype(int))
    # df.insert(pos_conc + 3, 'NOT_EM',       (df['TP_ST_CONCLUSAO'] == 4).astype(int))
    df = df.drop(columns=['TP_ST_CONCLUSAO'])

    # pos_esc = df.columns.get_loc('TP_ESCOLA') + 1
    # df.insert(pos_esc,     'ESC_PU',  (df['TP_ESCOLA'] == 2).astype(int))
    # df.insert(pos_esc + 1, 'ESC_PRI', (df['TP_ESCOLA'] == 3).astype(int))
    # df.insert(pos_esc + 2, 'ESC_EX',  (df['TP_ESCOLA'] == 4).astype(int))
    df = df.drop(columns=['TP_ESCOLA'])

    #pos_ens = df.columns.get_loc('TP_ENSINO') + 1
    # df.insert(pos_ens,     'ENSINO_REGULAR',      (df['TP_ENSINO'] == 1).astype(int))
    # df.insert(pos_ens + 1, 'EDUC_ESPECIAL',       (df['TP_ENSINO'] == 2).astype(int))
    # df.insert(pos_ens + 2, 'EDUC_JOVENS_ADULTOS', (df['TP_ENSINO'] == 3).astype(int))
    df = df.drop(columns=['TP_ENSINO'])

    df = df.drop(columns=['NO_MUNICIPIO_PROVA'])

    #pos_uf = df.columns.get_loc('SG_UF_PROVA') + 1
    #for i, uf in enumerate(ufs):
     #   df.insert(pos_uf + i, f'PROVA_IN_{uf}', (df['SG_UF_PROVA'] == uf).astype(int))
    df = df.drop(columns=['SG_UF_PROVA'])

    #pos_red = df.columns.get_loc('TP_STATUS_REDACAO') + 1
    #for i, (codigo, nome_col) in enumerate(status_redacao):
    #    df.insert(pos_red + i, nome_col, (df['TP_STATUS_REDACAO'] == codigo).astype(int))
    df = df.drop(columns=['TP_STATUS_REDACAO'])

    for col_origem, prefixo in [('Q001', 'PAI'), ('Q002', 'MAE')]:
        # pos = df.columns.get_loc(col_origem) + 1
        # for i, (letra, sufixo) in enumerate(escolaridade_map):
        #     df.insert(pos + i, f'{prefixo}_{sufixo}', (df[col_origem] == letra).astype(int))
        df = df.drop(columns=[col_origem])

    # for col_origem, prefixo in [('Q003', 'PAI'), ('Q004', 'MAE')]:
    #    pos = df.columns.get_loc(col_origem) + 1
    #    for i, (letra, sufixo) in enumerate(trabalho_map):
    #        df.insert(pos + i, f'{prefixo}_{sufixo}', (df[col_origem] == letra).astype(int))
    #    df = df.drop(columns=[col_origem])

    df['Q003'] = df['Q003'].map(mapa_trabalho_salario)
    df['Q004'] = df['Q004'].map(mapa_trabalho_salario)
    df = df.rename(columns={'Q003': 'PAI_TRABALHO', 'Q004': 'MAE_TRABALHO'})

    df = df.rename(columns={'Q005': 'QTD_MORADORES'})

    df['Q006'] = df['Q006'].map(mapa_renda)
    df = df.rename(columns={'Q006': 'RENDA_FAMILIAR'})

    df['Q007'] = df['Q007'].map(mapa_q007)
    df = df.rename(columns={'Q007': 'QTD_DIAS_EMPREGADA_DOMESTICA'})

    for col in renomear_qtd:
        df[col] = df[col].map(mapa_qtd_itens)
    df = df.rename(columns=renomear_qtd)

    for col in renomear_tem:
        df[col] = df[col].map(mapa_binario)
    df = df.rename(columns=renomear_tem)

    df = df.drop(columns=['NU_ANO'])

    df = df.drop(columns=['TP_LINGUA'])

    return df

# 3. Executa o sorteio de 1.000 exemplos e guarda também em matriz NumPy
print("Sorteando 1.000 participantes fixos... (isso leva alguns segundos)")
df_aleatorio = pegar_linhas_aleatorias(caminho_arquivo, n=10000, seed=42)
matriz = df_aleatorio.to_numpy()

print(f"Total carregado: {df_aleatorio.shape[0]} linhas e {df_aleatorio.shape[1]} colunas.")
display(df_aleatorio.head(1000))
#print(matriz.dtype, np.isnan(matriz).sum())
df_aleatorio.to_csv('/content/drive/MyDrive/ENEM_2019/NewDatasetENEM2019.csv', sep=';', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sorteando 1.000 participantes fixos... (isso leva alguns segundos)
Total carregado: 10000 linhas e 33 colunas.


,NU_INSCRICAO,TP_FAIXA_ETARIA,IS_FEM,TP_ANO_CONCLUIU,IN_TREINEIRO,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,PAI_TRABALHO,MAE_TRABALHO,QTD_MORADORES,RENDA_FAMILIAR,QTD_DIAS_EMPREGADA_DOMESTICA,QTD_BANHEIROS,QTD_QUARTOS_DE_DORMIR,QTD_CARRO,QTD_MOTO,QTD_GELADEIRA,QTD_FREEZER,QTD_LAVA_ROUPA,QTD_SECA_ROUPA,QTD_MICROONDAS,QTD_LAVA_LOUCA,TEM_ASPIRADOR_DE_PO,QTD_TV,TEM_DVD,TEM_TV_ASSINATURA,QTD_CELULAR,TEM_TELEFONE_FIXO,QTD_COMPUTADOR,TEM_INTERNET
0,190001605692,18,1,0,0,441.7,479.9,547.8,528.4,780,1350.0,0.0,4,1247.51,0.0,1,2,1,0,1,1,1,0,1,0,0,2,0,1,3,1,0,1
1,190003719845,17,1,0,1,466.7,419.3,483.1,386.1,680,1350.0,1350.0,2,1746.51,0.0,1,2,0,0,1,1,1,0,1,0,0,2,1,0,2,0,1,1
2,190005244929,19,1,2016,0,607.6,619.7,576.7,551.6,800,3050.0,3050.0,4,10978.01,1.5,3,3,2,0,1,1,1,0,1,1,1,1,1,0,1,1,1,1
3,190003122646,17,1,0,1,458.4,469.5,536.0,482.3,600,1350.0,1350.0,2,1247.51,0.0,1,2,0,1,1,0,0,0,1,0,0,1,0,0,2,0,0,1
4,190002144869,15,1,0,1,496.4,516.8,559.2,511.8,660,0.0,3050.0,4,1247.51,0.0,2,3,1,0,1,1,1,0,1,0,0,1,0,0,2,0,1,1
5,190002045756,18,0,0,0,451.4,404.3,345.5,462.5,420,1350.0,1350.0,3,499.00,0.0,1,3,0,0,2,0,0,0,0,0,0,2,1,1,3,0,0,1
6,190002988666,18,1,0,0,406.5,475.6,477.4,485.5,280,1119.0,1119.0,5,1247.51,0.0,1,2,0,0,1,0,0,0,1,0,0,0,0,0,3,1,0,1
7,190005751247,33,1,0,0,619.9,621.8,617.3,625.0,740,3050.0,5450.0,5,2245.51,0.0,2,3,0,0,1,1,1,0,1,0,0,1,1,0,3,0,0,1
8,190004256924,17,1,0,0,477.9,424.7,434.1,407.0,500,1350.0,0.0,3,1247.51,0.0,1,2,0,0,1,0,1,0,0,0,0,1,1,0,2,0,0,1
9,190003109558,17,0,0,0,596.9,527.7,553.8,596.7,920,1800.0,1350.0,4,499.00,0.0,1,3,0,1,1,0,0,0,0,0,0,1,0,0,2,0,0,0
